In [13]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import textstat
from tqdm import tqdm

# integrating tqdm with pandas (to see a progress bar)
tqdm.pandas()

# seeing every columns
pd.set_option('display.max_columns', None)

In [15]:
df = pd.read_csv("spotify_millsongdata.csv")
df = df.head(100) # only use 500 rows just for test
print(df.head())
print(df.info())

  artist                   song                                        link  \
0   ABBA  Ahe's My Kind Of Girl  /a/abba/ahes+my+kind+of+girl_20598417.html   
1   ABBA       Andante, Andante       /a/abba/andante+andante_20002708.html   
2   ABBA         As Good As New        /a/abba/as+good+as+new_20003033.html   
3   ABBA                   Bang                  /a/abba/bang_20598415.html   
4   ABBA       Bang-A-Boomerang      /a/abba/bang+a+boomerang_20002668.html   

                                                text  
0  Look at her face, it's a wonderful face  \r\nA...  
1  Take it easy with me, please  \r\nTouch me gen...  
2  I'll never know why I had to go  \r\nWhy I had...  
3  Making somebody happy is a question of give an...  
4  Making somebody happy is a question of give an...  
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   artist  100 non-null    str 

In [4]:
print("1. Step: Sentiment Analysis (Sentiment Analysis)")
# We are downloading and preparing the VADER emotion analysis tool of the NLTK library
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# We take the 'compound' score of the lyrics (between -1 and Dec 1)
df['sentiment_score'] = df['text'].progress_apply(lambda x: sia.polarity_scores(str(x))['compound'])

1. Step: Sentiment Analysis (Sentiment Analysis)


100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 344.26it/s]


In [5]:
df.head(20)

,artist,song,link,text,sentiment_score
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA...",0.9587
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen...",0.9877
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...,0.9986
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...,0.9971
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...,0.9974
5,ABBA,Burning My Bridges,/a/abba/burning+my+bridges_20003011.html,"Well, you hoot and you holler and you make me ...",-0.9382
6,ABBA,Cassandra,/a/abba/cassandra_20002811.html,Down in the street they're all singing and sho...,-0.9956
7,ABBA,Chiquitita,/a/abba/chiquitita_20002978.html,"Chiquitita, tell me what's wrong \r\nYou're e...",-0.9640
8,ABBA,Crazy World,/a/abba/crazy+world_20003013.html,I was out with the morning sun \r\nCouldn't s...,-0.2324
9,ABBA,Crying Over You,/a/abba/crying+over+you_20177611.html,I'm waitin' for you baby \r\nI'm sitting all ...,0.6904


In [6]:
print("\n2. Step: Cognitive Analysis (Flesch-Kincaid Readability)")
# we calculate the Flesch-Kincaid convenience score using the textstat library
# Note: This metric measures linguistic complexity based on word and sentence lengths.
df['readability_score'] = df['text'].progress_apply(lambda x: textstat.flesch_reading_ease(str(x)))


2. Step: Cognitive Analysis (Flesch-Kincaid Readability)


100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 100.51it/s]


In [7]:
df.head()

,artist,song,link,text,sentiment_score,readability_score
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA...",0.9587,80.670118
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen...",0.9877,-164.441923
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...,0.9986,-209.087308
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...,0.9971,-106.991000
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...,0.9974,-106.080455


In [8]:
print("\n3. Step: Semantic Analysis (TF-IDF + LSA)")

# Creating the TF-IDF Matrix (Stop words: we eliminate unnecessary words in English)
tfidf = TfidfVectorizer(stop_words='english', max_features=4000)
tfidf_matrix = tfidf.fit_transform(df['text'])

print(tfidf_matrix)


3. Step: Semantic Analysis (TF-IDF + LSA)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5096 stored elements and shape (100, 1907)>
  Coords	Values
  (0, 984)	0.162483942633298
  (0, 551)	0.18547512383601136
  (0, 1874)	0.14097200341306831
  (0, 1041)	0.14097200341306831
  (0, 1527)	0.11786527282035925
  (0, 1831)	0.07271018515098363
  (0, 1497)	0.15056216309477624
  (0, 1407)	0.15056216309477624
  (0, 1013)	0.15056216309477624
  (0, 591)	0.15056216309477624
  (0, 889)	0.24414339598926713
  (0, 900)	0.4067216038273597
  (0, 679)	0.3365703183601035
  (0, 1023)	0.20869740381871618
  (0, 584)	0.1354508008112957
  (0, 606)	0.22787772318213206
  (0, 119)	0.19841988028735308
  (0, 149)	0.2144986203654699
  (0, 935)	0.25491086500413435
  (0, 1808)	0.12231667073638564
  (0, 1170)	0.13353329765428196
  (0, 807)	0.1640787340057774
  (0, 1541)	0.1640787340057774
  (0, 743)	0.10434870190935809
  (0, 971)	0.0696308313252966
  :	:
  (99, 844)	0.09151912423345865
  (99, 802)	0.091519

In [9]:
# Latent Semantic Analysis (LSA) - Dimension Reduction
n_components = 10
lsa = TruncatedSVD(n_components=n_components, random_state=42)
lsa_matrix = lsa.fit_transform(tfidf_matrix)

# Adding LSA results as columns to the main DataFrame
lsa_columns = [f'lsa_dim_{i+1}' for i in range(n_components)]
df_lsa = pd.DataFrame(lsa_matrix, columns=lsa_columns, index=df.index)
df = pd.concat([df, df_lsa], axis=1)

In [10]:
df.head()

,artist,song,link,text,sentiment_score,readability_score,lsa_dim_1,lsa_dim_2,lsa_dim_3,lsa_dim_4,lsa_dim_5,lsa_dim_6,lsa_dim_7,lsa_dim_8,lsa_dim_9,lsa_dim_10
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA...",0.9587,80.670118,0.273002,0.035855,0.042588,-0.129640,0.042439,-0.189922,0.052368,0.352751,0.141857,-0.016668
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen...",0.9877,-164.441923,0.226479,-0.093496,0.109798,0.065914,-0.035046,0.234119,-0.054166,-0.095665,-0.083342,-0.144907
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...,0.9986,-209.087308,0.126539,0.001779,-0.066553,-0.073476,-0.039724,0.045848,0.050934,-0.000202,-0.023327,0.032438
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...,0.9971,-106.991000,0.147256,-0.031944,-0.161112,-0.135041,-0.104660,0.095265,0.386001,-0.273463,0.277137,-0.172519
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...,0.9974,-106.080455,0.174051,-0.037841,-0.181597,-0.146920,-0.133934,0.112003,0.455595,-0.344802,0.403008,-0.255784


In [11]:
print("\n4. Step: Data Standardization")
features_to_scale = ['sentiment_score', 'readability_score'] + lsa_columns

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features_to_scale])

# Adding scaled data as new columns (prefixing them with 'scaled_')
scaled_columns = [f'scaled_{col}' for col in features_to_scale]
df_scaled = pd.DataFrame(scaled_features, columns=scaled_columns, index=df.index)
df = pd.concat([df, df_scaled], axis=1)


4. Step: Data Standardization


In [14]:
df.head()

,artist,song,link,text,sentiment_score,readability_score,lsa_dim_1,lsa_dim_2,lsa_dim_3,lsa_dim_4,lsa_dim_5,lsa_dim_6,lsa_dim_7,lsa_dim_8,lsa_dim_9,lsa_dim_10,scaled_sentiment_score,scaled_readability_score,scaled_lsa_dim_1,scaled_lsa_dim_2,scaled_lsa_dim_3,scaled_lsa_dim_4,scaled_lsa_dim_5,scaled_lsa_dim_6,scaled_lsa_dim_7,scaled_lsa_dim_8,scaled_lsa_dim_9,scaled_lsa_dim_10
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA...",0.9587,80.670118,0.273002,0.035855,0.042588,-0.129640,0.042439,-0.189922,0.052368,0.352751,0.141857,-0.016668,0.610934,1.570321,1.006208,0.198756,0.338950,-0.896978,0.291951,-1.385757,0.456391,2.717305,1.005403,-0.062536
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen...",0.9877,-164.441923,0.226479,-0.093496,0.109798,0.065914,-0.035046,0.234119,-0.054166,-0.095665,-0.083342,-0.144907,0.649945,-0.544311,0.397205,-0.594699,0.794244,0.481500,-0.261855,1.714444,-0.327487,-0.767036,-0.795166,-1.109793
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...,0.9986,-209.087308,0.126539,0.001779,-0.066553,-0.073476,-0.039724,0.045848,0.050934,-0.000202,-0.023327,0.032438,0.664608,-0.929476,-0.911073,-0.010272,-0.400404,-0.501069,-0.295285,0.337979,0.445835,-0.025256,-0.315323,0.338482
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...,0.9971,-106.991000,0.147256,-0.031944,-0.161112,-0.135041,-0.104660,0.095265,0.386001,-0.273463,0.277137,-0.172519,0.662590,-0.048670,-0.639872,-0.217136,-1.040970,-0.935049,-0.759400,0.699270,2.911246,-2.148587,2.087025,-1.335279
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...,0.9974,-106.080455,0.174051,-0.037841,-0.181597,-0.146920,-0.133934,0.112003,0.455595,-0.344802,0.403008,-0.255784,0.662993,-0.040815,-0.289118,-0.253308,-1.179745,-1.018781,-0.968630,0.821642,3.423313,-2.702914,3.093420,-2.015260
